# FINAL ImpactStudio FullSystem_v6

Minimal Gemini notebook for FastAPI script review testing.


## Run Order

1. Install dependencies.
2. Load `GOOGLE_API_KEY`.
3. Define the script reviewer prompt.
4. Run the Gemini smoke test.
5. Use `SECTION_E_API_SERVER.py` for FastAPI serving.


In [22]:
# STEP 1: Install dependencies
print("[STEP 1] Installing dependencies...")
!pip -q install -U google-genai fastapi uvicorn python-multipart pyngrok pypdf python-docx
print("[STEP 1] Done.")


[STEP 1] Installing dependencies...
[STEP 1] Done.


In [23]:
# STEP 2: Imports
print("[STEP 2] Importing libraries...")
import os
import re
import json
import getpass
from google import genai
print("[STEP 2] Imports ready.")


[STEP 2] Importing libraries...
[STEP 2] Imports ready.


In [ ]:
# STEP 3: Load GOOGLE_API_KEY
if not os.getenv("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter GOOGLE_API_KEY: ")

def _normalize_google_api_key(raw: str) -> str:
    raw = (raw or "").strip()
    match = re.search(r"AIza[0-9A-Za-z_-]{20,}", raw)
    return match.group(0).strip() if match else raw

GOOGLE_API_KEY = _normalize_google_api_key(os.getenv("GOOGLE_API_KEY", ""))
os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
if not GOOGLE_API_KEY:
    raise ValueError("GOOGLE_API_KEY is empty.")
print(f"[STEP 3] GOOGLE_API_KEY ready: {GOOGLE_API_KEY[:8]}...")


[STEP 3] GOOGLE_API_KEY ready: AIzaSyCV...


In [25]:
# STEP 4: Script reviewer config
GEMINI_MODEL_NAME = os.getenv("GEMINI_MODEL_NAME", "gemini-2.5-flash-lite")
SCRIPT_REVIEWER_SYSTEM_PROMPT = """You are Script Reviewer for Impact Studios.
Return only valid JSON with keys verdict, score, benefits, risks, rationale.
verdict must be Yes or No. score must be 1 to 5.
benefits and risks must be arrays of concise strings.
rationale must be a short paragraph suitable for direct frontend display.
""".strip()

SCRIPT_REVIEWER_USER_PROMPT = """Mission Context:
{mission_text}

User Prompt:
{user_prompt}

Submitted Material:
{submission_text}

Task:
Review this material and produce the required JSON output.
""".strip()

print("[STEP 4] Script reviewer prompt ready.")


[STEP 4] Script reviewer prompt ready.


In [26]:
# STEP 5: Gemini helper
client = genai.Client(api_key=GOOGLE_API_KEY)

def build_user_prompt(user_prompt: str, submission_text: str, mission_text: str = "") -> str:
    return SCRIPT_REVIEWER_USER_PROMPT.format(
        mission_text=(mission_text or "").strip() or "[No mission provided]",
        user_prompt=(user_prompt or "").strip() or "[No extra user prompt provided]",
        submission_text=(submission_text or "").strip(),
    )

def call_script_reviewer(user_prompt: str, submission_text: str, mission_text: str = "") -> dict:
    prompt_text = build_user_prompt(user_prompt, submission_text, mission_text)
    response = client.models.generate_content(
        model=GEMINI_MODEL_NAME,
        config={
            "system_instruction": SCRIPT_REVIEWER_SYSTEM_PROMPT,
            "temperature": 0.2,
            "response_mime_type": "application/json",
        },
        contents=prompt_text,
    )
    text = (getattr(response, "text", "") or "").strip()
    result = json.loads(text)
    result["type"] = "review"
    return result

print("[STEP 5] Gemini helper ready.")


[STEP 5] Gemini helper ready.


In [27]:
# STEP 6: Smoke test
TEST_USER_PROMPT = "Review this screenplay for uplifting value."
TEST_SUBMISSION = "A tense family reunion slowly turns into an honest conversation about grief, accountability, and repair."
TEST_MISSION = "Assess whether the narrative feels constructive, emotionally clear, and audience-ready."

smoke_result = call_script_reviewer(TEST_USER_PROMPT, TEST_SUBMISSION, TEST_MISSION)
print("[STEP 6] Smoke test result:")
print(json.dumps(smoke_result, indent=2, ensure_ascii=False))


[STEP 6] Smoke test result:
{
  "verdict": "Yes",
  "score": 4,
  "benefits": [
    "Explores universal themes of grief and family dynamics.",
    "Offers potential for emotional catharsis and healing.",
    "Provides opportunities for strong character development.",
    "Can resonate with a wide audience seeking meaningful stories."
  ],
  "risks": [
    "Potential for the tone to become overly somber if not balanced.",
    "Requires sensitive handling to avoid melodrama.",
    "Pacing needs careful management to maintain audience engagement during dialogue-heavy scenes."
  ],
  "rationale": "The screenplay presents a compelling narrative arc that moves from tension to resolution through honest conversation. Its exploration of grief, accountability, and repair offers significant uplifting value, providing a cathartic and relatable experience for the audience. With careful execution, particularly in balancing emotional depth with narrative momentum, this story is well-positioned for au

## Section E - FastAPI Backend

Run the next three cells in Colab to expose the Gemini script reviewer through FastAPI.


In [28]:
# STEP 7: Install FastAPI dependencies for Colab backend
print("[API] Installing server dependencies...")

!pip -q install fastapi uvicorn python-multipart pyngrok pypdf python-docx

print("[API] Dependency install complete.")


[API] Installing server dependencies...
[API] Dependency install complete.


In [29]:
# STEP 8: Define orchestrated FastAPI server (v2 — Orchestrator + 3 Agents)
# Agents are strictly isolated: each has its own system prompt and output schema.
# Routes: chat | script_review | impact_analysis
print("[API] Defining orchestrated ImpactStudio server...")

import os, json, time, asyncio, tempfile, traceback, uuid

from fastapi import FastAPI, UploadFile, File, Form
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import StreamingResponse
from google import genai
import uvicorn

GEMINI_API_KEY = globals().get("GOOGLE_API_KEY", os.getenv("GOOGLE_API_KEY", "")).strip()
GEMINI_MODEL_NAME = globals().get("GEMINI_MODEL_NAME", os.getenv("GEMINI_MODEL_NAME", "gemini-2.5-flash-lite"))

# ── RAG Setup (optional — graceful fallback if sentence-transformers unavailable) ──
try:
    from sentence_transformers import SentenceTransformer
    import numpy as np
    _ST_AVAILABLE = True
    print("[API] sentence-transformers available — RAG enabled")
except ImportError:
    _ST_AVAILABLE = False
    print("[API] sentence-transformers not installed — RAG disabled (run: pip install sentence-transformers)")

class SimpleRAG:
    def __init__(self, model_name="all-MiniLM-L6-v2"):
        self.chunks = []
        self.embeddings = []
        self.model = None
        if _ST_AVAILABLE:
            try:
                self.model = SentenceTransformer(model_name)
                print(f"[RAG] Embedding model loaded: {model_name}")
            except Exception as e:
                print(f"[RAG] Could not load embedding model: {e}")

    def add_document(self, text, source="unknown", chunk_size=800, overlap=120):
        words = text.split()
        for i in range(0, len(words), chunk_size - overlap):
            chunk_text = " ".join(words[i:i + chunk_size])
            if len(chunk_text.strip()) < 50:
                continue
            chunk = {"text": chunk_text, "source": source, "chunk_id": len(self.chunks)}
            self.chunks.append(chunk)
            if self.model:
                import numpy as np
                emb = self.model.encode(chunk_text, convert_to_numpy=True)
                self.embeddings.append(emb)
        print(f"[RAG] Added {source}: {len(self.chunks)} total chunks")

    def search(self, query, k=4):
        if not self.chunks or not self.model or not self.embeddings:
            return []
        import numpy as np
        q_emb = self.model.encode(query, convert_to_numpy=True)
        scores = [
            float(np.dot(q_emb, e) / (np.linalg.norm(q_emb) * np.linalg.norm(e) + 1e-9))
            for e in self.embeddings
        ]
        top = sorted(enumerate(scores), key=lambda x: x[1], reverse=True)[:k]
        return [(scores[i], self.chunks[i]) for i, _ in top]

rag = SimpleRAG()

def _render_rag_context(hits):
    if not hits:
        return "[No reference context available — RAG index is empty]"
    lines = []
    for rank, (score, ch) in enumerate(hits, 1):
        header = f"[Ref {rank}] (score={score:.3f}) source={os.path.basename(ch['source'])}"
        lines.append(header + "\n" + ch["text"])
    return "\n\n".join(lines)

# ── Shared Utilities ──────────────────────────────────────────────────────────

def _api_read_file(file_path: str) -> str:
    if not file_path:
        return ""
    ext = os.path.splitext(file_path)[1].lower()
    try:
        if ext == ".pdf":
            from pypdf import PdfReader
            return "\n".join(p.extract_text() or "" for p in PdfReader(file_path).pages)
        if ext == ".docx":
            from docx import Document as DocxDocument
            doc = DocxDocument(file_path)
            return "\n".join(p.text for p in doc.paragraphs)
        if ext in [".txt", ".md"]:
            with open(file_path, "r", encoding="utf-8", errors="replace") as f:
                return f.read()
        return f"ERROR: Unsupported file type: {ext}"
    except Exception as e:
        return f"ERROR: {e}"

def _combine_user_input(story_text: str, uploaded_text: str) -> str:
    s = (story_text or "").strip()
    u = (uploaded_text or "").strip()
    if s and u:
        return f"{s}\n\n[Uploaded File Content]\n{u}"
    return s or u

def _build_memory_context(chat_history, max_turns=4, max_chars=2400, max_msg=360):
    flat = []
    for msg in (chat_history or []):
        role = msg.get("role", "")
        content = (msg.get("content") or "").strip()
        if not content or role not in ("user", "assistant"):
            continue
        if len(content) > max_msg:
            content = content[:max_msg - 3] + "..."
        flat.append(f"{'User' if role == 'user' else 'Assistant'}: {content}")
    if not flat:
        return "[No prior conversation]"
    recent = flat[-(max_turns * 2):]
    memory = "\n\n".join(recent)
    if len(memory) > max_chars:
        memory = "...\n" + memory[-(max_chars - 4):]
    return memory

def _extract_json(text: str) -> str:
    raw = (text or "").strip().strip("`")
    if raw.startswith("json"):
        raw = raw[4:].strip()
    s, e = raw.find("{"), raw.rfind("}")
    return raw[s:e + 1] if s != -1 and e > s else raw

def _sse(data: dict) -> str:
    return f"data: {json.dumps(data, ensure_ascii=False)}\n\n"

# ── Orchestrator (3-way routing) ──────────────────────────────────────────────
# Strictly a router — does NOT process content itself.

_ORCHESTRATOR_SYSTEM_PROMPT = """You are a routing system. Classify the user input into exactly one of three categories:

CHAT — The user is having a conversation: asking a follow-up question, referencing a prior result, greeting, or chatting without submitting new creative content for analysis.

SCRIPT_REVIEW — The user is submitting creative writing, a screenplay, or a document and wants a structured quality critique with a verdict and score.

IMPACT_ANALYSIS — The user is submitting content and wants to understand its community impact, who it affects, and what ethical outreach actions the creator should take.

Respond with ONLY one word: CHAT, SCRIPT_REVIEW, or IMPACT_ANALYSIS."""

# def _orchestrate(user_prompt: str, has_file: bool, memory_text: str) -> str:
#     """Returns 'chat' | 'script_review' | 'impact_analysis'. Never raises."""
#     combined = (user_prompt or "").strip().lower()

#     # Fast conversational signal detection → chat (no LLM call needed)
#     chat_signals = [
#         "what score", "what did you", "tell me what", "you gave", "last time",
#         "you said", "how did", "remind me", "what was", "thanks", "thank you",
#         "that's great", "got it", "can you explain", "what do you mean",
#     ]
#     if any(sig in combined for sig in chat_signals) and not has_file:
#         return "chat"

#     # Question-type detection: prompts that start with question words are rarely
#     # direct script-review requests — they ask ABOUT content or prior results.
#     question_starters = (
#         "what ", "who ", "how ", "why ", "which ", "does ", "do ",
#         "is ", "are ", "can ", "could ", "would ", "should ",
#         "tell me", "explain",
#     )
#     is_question = any(combined.startswith(q) for q in question_starters)

#     # Keyword banks — intentionally EXCLUDE generic content words like
#     # "screenplay" / "script" / "story" that appear in impact questions too.
#     impact_kw = [
#         "impact", "community", "communities", "outreach", "ethics", "harm",
#         "harmed", "harmful", "society", "who is affected", "who might",
#         "who could", "vulnerable", "support", "advocacy", "audience",
#         "affect", "affects", "affected", "themes", "ethical",
#     ]
#     # script_kw covers clear REVIEW INTENT signals only (not content-type words)
#     script_kw = [
#         "review", "critique", "uplifting", "analyze this", "read this",
#         "feedback on", "give me a score", "rate this", "evaluate this",
#     ]

#     impact_score = sum(1 for k in impact_kw if k in combined)
#     script_score = sum(1 for k in script_kw if k in combined)

#     if has_file:
#         # File upload: impact keywords win, otherwise default script_review
#         return "impact_analysis" if impact_score > script_score else "script_review"

#     # Questions with impact signals → impact_analysis (even without a file)
#     if is_question and impact_score > 0:
#         return "impact_analysis"

#     if impact_score > 0 and script_score == 0:
#         return "impact_analysis"
#     if script_score > 0 and impact_score == 0:
#         return "script_review"
#     if not combined:
#         return "chat"

#     # Ambiguous: LLM routing call
#     try:
#         client = genai.Client(api_key=GEMINI_API_KEY)
#         routing_input = f"Memory:\n{memory_text}\n\nUser input: {user_prompt}\nFile attached: {has_file}"
#         response = client.models.generate_content(
#             model=GEMINI_MODEL_NAME,
#             config={"system_instruction": _ORCHESTRATOR_SYSTEM_PROMPT, "temperature": 0},
#             contents=routing_input,
#         )
#         decision = (getattr(response, "text", "") or "").strip().upper()
#         if "IMPACT" in decision:
#             return "impact_analysis"
#         if "SCRIPT" in decision:
#             return "script_review"
#         return "chat"
#     except Exception:
#         return "script_review" if has_file else "chat"

# ── Agent 1: Chat Handler ─────────────────────────────────────────────────────
# Handles conversational follow-ups. Does NOT perform reviews or analysis.

_CHAT_SYSTEM_PROMPT = """You are ImpactStudio's conversational assistant.

You have access to recent conversation memory shown below. Use it to answer follow-up questions naturally and accurately.

Rules:
- If the user references a prior review score, verdict, or result, cite it directly from memory.
- Do NOT perform a new script review or impact analysis. Do NOT output JSON or structured data.
- Keep responses concise (under 150 words) unless the user explicitly asks for more detail.
- Write in plain prose. No markdown headers. No bullet points unless the user asks.
- Be warm and professional."""

def _call_chat_handler(user_prompt: str, memory_text: str) -> dict:
    client = genai.Client(api_key=GEMINI_API_KEY)
    prompt = f"Conversation Memory:\\n{memory_text}\\n\\nUser: {user_prompt}"
    response = client.models.generate_content(
        model=GEMINI_MODEL_NAME,
        config={"system_instruction": _CHAT_SYSTEM_PROMPT, "temperature": 0.4},
        contents=prompt,
    )
    message = (getattr(response, "text", "") or "").strip()
    return {"type": "chat", "message": message or "I need a bit more context to answer that."}

def _orchestrate(user_prompt: str, has_file: bool, memory_text: str) -> str:
    """Returns 'chat' | 'script_review' | 'impact_analysis'. Never raises."""
    combined = (user_prompt or "").strip().lower()

    chat_signals = [
        "what score", "what did you", "tell me what", "you gave", "last time",
        "you said", "how did", "remind me", "what was", "thanks", "thank you",
        "that's great", "got it", "can you explain", "what do you mean",
    ]
    if any(sig in combined for sig in chat_signals) and not has_file:
        return "chat"

    question_starters = (
        "what ", "who ", "how ", "why ", "which ", "does ", "do ",
        "is ", "are ", "can ", "could ", "would ", "should ",
        "tell me", "explain",
    )
    is_question = any(combined.startswith(q) for q in question_starters)

    impact_kw = [
        "impact", "community", "communities", "outreach", "ethics", "harm",
        "harmed", "harmful", "society", "who is affected", "who might",
        "who could", "vulnerable", "support", "advocacy", "audience",
        "affect", "affects", "affected", "themes", "ethical",
    ]
    script_kw = [
        "review", "critique", "uplifting", "analyze this", "read this",
        "feedback on", "give me a score", "rate this", "evaluate this",
    ]

    impact_score = sum(1 for k in impact_kw if k in combined)
    script_score = sum(1 for k in script_kw if k in combined)

    if has_file:
        return "impact_analysis" if impact_score > script_score else "script_review"

    if is_question and impact_score > 0:
        return "impact_analysis"

    if impact_score > 0 and script_score == 0:
        return "impact_analysis"
    if script_score > 0 and impact_score == 0:
        return "script_review"
    if not combined:
        return "chat"

    try:
        client = genai.Client(api_key=GEMINI_API_KEY)
        routing_input = f"Memory:\n{memory_text}\n\nUser input: {user_prompt}\nFile attached: {has_file}"
        response = client.models.generate_content(
            model=GEMINI_MODEL_NAME,
            config={"system_instruction": _ORCHESTRATOR_SYSTEM_PROMPT, "temperature": 0},
            contents=routing_input,
        )
        decision = (getattr(response, "text", "") or "").strip().upper()
        if "IMPACT" in decision:
            return "impact_analysis"
        if "SCRIPT" in decision:
            return "script_review"
        return "chat"
    except Exception:
        return "script_review" if has_file else "chat"

# 验证
tests = [
    ("What communities does this screenplay affect?", False),
    ("Who might be harmed by the themes in this story?", False),
    ("You gave it a 4 — can you remind me why?", False),
    ("Please review this for uplifting value.", False),
]
for p, f in tests:
    print(f"[{_orchestrate(p, f, '')}] {p}")



# ── Agent 2: Script Reviewer ──────────────────────────────────────────────────
# Critiques creative writing. Returns structured verdict + score.

_SCRIPT_REVIEWER_SYSTEM_PROMPT = """You are Script Reviewer for Impact Studios.
Review creative writing or script material and return only valid JSON.

Schema:
{
  "verdict": "Yes" or "No",
  "score": integer 1-5,
  "benefits": ["string", ...],
  "risks": ["string", ...],
  "rationale": "string"
}

Rules:
- Return JSON only. No markdown. No code fences. No extra commentary.
- verdict must be "Yes" if score >= 3, otherwise "No".
- benefits: 2-4 concise strengths of the material.
- risks: 2-4 concise concerns or revision points.
- rationale: short paragraph suitable for direct frontend display.
- Focus on clarity, emotional effect, structure, and uplifting quality."""

_SCRIPT_REVIEWER_USER_PROMPT = """Mission Context:
{mission_text}

Conversation Memory:
{memory_text}

User Prompt:
{user_prompt}

Submitted Material:
{submission_text}

Task: Review this material and produce the required JSON output."""

def _call_script_reviewer(user_prompt: str, submission_text: str, mission_text: str, memory_text: str) -> dict:
    client = genai.Client(api_key=GEMINI_API_KEY)
    prompt = _SCRIPT_REVIEWER_USER_PROMPT.format(
        mission_text=(mission_text or "").strip() or "[No mission provided]",
        memory_text=memory_text,
        user_prompt=(user_prompt or "").strip() or "[No prompt provided]",
        submission_text=(submission_text or "").strip(),
    )
    response = client.models.generate_content(
        model=GEMINI_MODEL_NAME,
        config={
            "system_instruction": _SCRIPT_REVIEWER_SYSTEM_PROMPT,
            "temperature": 0.2,
            "response_mime_type": "application/json",
        },
        contents=prompt,
    )
    text = (getattr(response, "text", "") or "").strip()
    try:
        data = json.loads(_extract_json(text))
    except Exception:
        data = {"score": 3, "rationale": text, "benefits": [], "risks": []}

    score = max(1, min(5, int(data.get("score", 3))))
    verdict = "Yes" if score >= 3 else "No"
    benefits = [str(x).strip() for x in (data.get("benefits") or []) if str(x).strip()][:4]
    risks = [str(x).strip() for x in (data.get("risks") or []) if str(x).strip()][:4]
    rationale = str(data.get("rationale", "")).strip() or text

    if not benefits:
        benefits = ["Shows a workable creative core.", "Contains material that can be developed further."]
    if not risks:
        risks = ["Needs more specific revision detail.", "Would benefit from clearer structural refinement."]

    return {
        "type": "review",
        "verdict": verdict,
        "score": score,
        "benefits": benefits,
        "risks": risks,
        "rationale": rationale,
    }

# ── Agent 3: Impact Agent ─────────────────────────────────────────────────────
# Advisory agent. Identifies community impact and outreach recommendations.
# Uses RAG for reference context when available.

_IMPACT_AGENT_SYSTEM_PROMPT = """You are an editorial advisor focused on community engagement and ethical publication for Impact Studios.

Read the submitted creative writing and:
1. Identify communities, audiences, or individuals who may be directly affected by its themes.
2. For each topic: provide a direct verbatim quote from the submission as evidence, then give 2-3 specific, actionable outreach or support recommendations.
3. Provide an overall advisory note.

Return only valid JSON matching this exact schema:
{
  "communities": ["string", ...],
  "topics": [
    {
      "topic": "string",
      "quote": "string",
      "recommendations": ["string", "string", ...]
    }
  ],
  "overall_note": "string"
}

Rules:
- Return JSON only. No markdown. No code fences.
- communities: 2-5 specific affected groups (not generic like "everyone").
- topics: 2-4 topics identified from the content.
- quote: must be a verbatim excerpt from the submitted material.
- recommendations: 2-3 items per topic. Name specific organizations, exact resources, or concrete actions. Avoid vague advice like "be sensitive".
- overall_note: 2-3 sentences summarizing the advisory."""

_IMPACT_AGENT_USER_PROMPT = """Mission Context:
{mission_text}

Conversation Memory:
{memory_text}

Advisory Reference Context (RAG):
{rag_context}

Submitted Material:
{submission_text}

Task: Analyze the community impact of this material and produce the required JSON advisory output."""

def _call_impact_agent(submission_text: str, mission_text: str, memory_text: str) -> dict:
    client = genai.Client(api_key=GEMINI_API_KEY)

    # RAG retrieval (optional)
    hits = rag.search(submission_text, k=4) if rag.chunks else []
    rag_context = _render_rag_context(hits)
    sources = [
        f"Chunk {i+1} — {os.path.basename(ch['source'])}#{ch['chunk_id']}"
        for i, (_, ch) in enumerate(hits)
    ] if hits else []

    prompt = _IMPACT_AGENT_USER_PROMPT.format(
        mission_text=(mission_text or "").strip() or "[No mission provided]",
        memory_text=memory_text,
        rag_context=rag_context,
        submission_text=(submission_text or "").strip(),
    )
    response = client.models.generate_content(
        model=GEMINI_MODEL_NAME,
        config={
            "system_instruction": _IMPACT_AGENT_SYSTEM_PROMPT,
            "temperature": 0.3,
            "response_mime_type": "application/json",
        },
        contents=prompt,
    )
    text = (getattr(response, "text", "") or "").strip()
    try:
        data = json.loads(_extract_json(text))
    except Exception:
        data = {"communities": [], "topics": [], "overall_note": text}

    communities = [str(c).strip() for c in (data.get("communities") or []) if str(c).strip()][:5]
    topics = []
    for t in (data.get("topics") or [])[:4]:
        topic_name = str(t.get("topic", "")).strip()
        quote = str(t.get("quote", "")).strip()
        recs = [str(r).strip() for r in (t.get("recommendations") or []) if str(r).strip()][:3]
        if topic_name and quote and recs:
            topics.append({"topic": topic_name, "quote": quote, "recommendations": recs})
    overall_note = str(data.get("overall_note", "")).strip() or "No advisory note generated."

    if not communities:
        communities = ["General audience"]
    if not topics:
        topics = [{
            "topic": "General Themes",
            "quote": submission_text[:200].strip(),
            "recommendations": ["Consider community consultation before publication."]
        }]

    return {
        "type": "impact_analysis",
        "communities": communities,
        "topics": topics,
        "overall_note": overall_note,
        "sources": sources,
    }

# ── FastAPI App ───────────────────────────────────────────────────────────────

app = FastAPI(title="ImpactStudio Orchestrated API", version="4.0")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

@app.get("/api/health")
def health_check():
    return {
        "status": "ok",
        "model": GEMINI_MODEL_NAME,
        "rag_chunks": len(rag.chunks),
        "agents": ["chat", "script_review", "impact_analysis"],
    }

@app.post("/api/analyze")
async def analyze_endpoint(
    story_text: str = Form(""),
    mission_text: str = Form(""),
    file: UploadFile = File(None),
    chat_history: str = Form("[]"),
):
    async def event_stream():
        uploaded_path = ""
        try:
            try:
                history = json.loads(chat_history)
            except Exception:
                history = []

            memory_text = _build_memory_context(history)

            # Read uploaded file
            uploaded_text = ""
            uploaded_name = ""
            if file and file.filename:
                uploaded_name = file.filename
                suffix = os.path.splitext(file.filename)[1]
                tmp = tempfile.NamedTemporaryFile(delete=False, suffix=suffix)
                tmp.write(await file.read())
                tmp.close()
                uploaded_path = tmp.name
                uploaded_text = _api_read_file(tmp.name)
                if uploaded_text.startswith("ERROR:"):
                    yield _sse({"stage": "error", "msg": uploaded_text})
                    return

            has_file = bool(uploaded_name)
            merged = _combine_user_input(story_text, uploaded_text)

            if not merged.strip() and not (story_text or "").strip():
                yield _sse({"stage": "error", "msg": "Please enter text or upload a file."})
                return

            # ── Routing ──────────────────────────────────────────────────────
            yield _sse({"stage": "routing", "msg": "Routing your request..."})
            await asyncio.sleep(0.05)

            agent = _orchestrate(story_text, has_file, memory_text)
            agent_labels = {
                "chat": "Assistant",
                "script_review": "Script Reviewer",
                "impact_analysis": "Impact Agent",
            }
            agent_label = agent_labels[agent]

            yield _sse({
                "stage": "routed",
                "agent": agent_label,
                "route_mode": agent,
                "msg": f"Directing to {agent_label}.",
            })
            await asyncio.sleep(0.05)

            yield _sse({"stage": "working", "agent": agent_label, "msg": f"Running {agent_label}..."})

            # ── Dispatch ──────────────────────────────────────────────────────
            if agent == "chat":
                result = _call_chat_handler(story_text, memory_text)

            elif agent == "script_review":
                if not merged.strip():
                    yield _sse({"stage": "error", "msg": "No content to review. Please enter text or upload a file."})
                    return
                result = _call_script_reviewer(story_text, merged, mission_text, memory_text)

            else:  # impact_analysis
                if not merged.strip():
                    yield _sse({"stage": "error", "msg": "No content to analyze. Please enter text or upload a file."})
                    return
                result = _call_impact_agent(merged, mission_text, memory_text)

            yield _sse({
                "stage": "done",
                "agent": agent_label,
                "route_mode": agent,
                "result": result,
            })

        except Exception as e:
            traceback.print_exc()
            yield _sse({"stage": "error", "msg": f"Analysis failed: {str(e)}"})
        finally:
            if uploaded_path and os.path.exists(uploaded_path):
                try:
                    os.unlink(uploaded_path)
                except OSError:
                    pass

    return StreamingResponse(
        event_stream(),
        media_type="text/event-stream",
        headers={
            "Cache-Control": "no-cache",
            "Connection": "keep-alive",
            "X-Accel-Buffering": "no",
        },
    )

print("[API] Orchestrated FastAPI app defined (v4.0)")
print("[API] Agents: chat | script_review | impact_analysis")
print("[API] RAG: ready (index empty until documents added via rag.add_document())")
print("[API] Endpoints: GET /api/health, POST /api/analyze")


[API] Defining orchestrated ImpactStudio server...
[API] sentence-transformers not installed — RAG disabled (run: pip install sentence-transformers)
[impact_analysis] What communities does this screenplay affect?
[impact_analysis] Who might be harmed by the themes in this story?
[chat] You gave it a 4 — can you remind me why?
[script_review] Please review this for uplifting value.
[API] Orchestrated FastAPI app defined (v4.0)
[API] Agents: chat | script_review | impact_analysis
[API] RAG: ready (index empty until documents added via rag.add_document())
[API] Endpoints: GET /api/health, POST /api/analyze


In [30]:
import getpass
from pyngrok import ngrok
ngrok.set_auth_token(getpass.getpass("Enter ngrok auth token: "))
# 3AjTZgDGqDjXl9q2oiGMrEABBJI_4kvVBUfhzHimN99zeT9Cc

In [31]:
# STEP 9: Launch FastAPI server
print("[API] Starting server...")

import threading

_server_thread = None
_tunnel_url = None


def _start_server():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="warning")


_server_thread = threading.Thread(target=_start_server, daemon=True)
_server_thread.start()
time.sleep(2)
print("[API] Uvicorn running on port 8000.")

try:
    from pyngrok import ngrok

    _tunnel_url = str(ngrok.connect(8000, "http"))
    print(f"\n{'=' * 60}")
    print("  API IS LIVE")
    print(f"  Public URL: {_tunnel_url}")
    print(f"  Health:     {_tunnel_url}/api/health")
    print(f"  Analyze:    POST {_tunnel_url}/api/analyze")
    print(f"{'=' * 60}\n")
except Exception as e:
    print(f"[API] ngrok tunnel failed: {e}")
    print("[API] The API is still available on http://localhost:8000")

API_PUBLIC_URL = _tunnel_url or "http://localhost:8000"
print(f"\n[API] API_PUBLIC_URL = {API_PUBLIC_URL}")


[API] Starting server...


ERROR:    [Errno 48] error while attempting to bind on address ('0.0.0.0', 8000): address already in use


[API] Uvicorn running on port 8000.

  API IS LIVE
  Public URL: NgrokTunnel: "https://exudative-horacio-gadfly.ngrok-free.dev" -> "http://localhost:8000"
  Health:     NgrokTunnel: "https://exudative-horacio-gadfly.ngrok-free.dev" -> "http://localhost:8000"/api/health
  Analyze:    POST NgrokTunnel: "https://exudative-horacio-gadfly.ngrok-free.dev" -> "http://localhost:8000"/api/analyze


[API] API_PUBLIC_URL = NgrokTunnel: "https://exudative-horacio-gadfly.ngrok-free.dev" -> "http://localhost:8000"


chat

- Can you remind me why you gave this a low score last time?
- Thanks, can you explain that in simpler words?
- What was your main point from the previous result?

script_review

- Please review this script for uplifting value and give me a score.
- Critique this screenplay’s structure and strengths/risks.
- Rate this draft and give specific revision feedback.

impact_analysis

- Who might be affected by this story, and what ethical outreach actions should we take?
- Analyze the community impact of this script and suggest concrete support steps.
- What harms could this narrative cause, and how should we mitigate them responsibly?